In [ ]:
import sys

sys.path.append("../")

import scipy
import optuna

import numpy as np
import pandas as pd

from docplex.mp.model import Model
from qiskit.circuit import ClassicalRegister, ParameterVector, QuantumCircuit, QuantumRegister
from qiskit.primitives import BackendEstimatorV2
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_histogram
from qiskit_addon_opt_mapper.converters import OptimizationProblemToQubo
from qiskit_addon_opt_mapper.translators import from_docplex_mp
from qiskit_aer import AerSimulator

from dicke_state_ansatz import DickeStateAnsatz
from utils import objective_scipy, ObjectiveOptuna

In [ ]:
simulator = AerSimulator(method="matrix_product_state", device="CPU")
estimator = BackendEstimatorV2(backend=simulator)
pm = generate_preset_pass_manager(backend=simulator, optimization_level=0)

In [ ]:
assets_data = pd.read_csv(filepath_or_buffer="../datasets/sp500_assets_close_price.csv", index_col=0)

In [ ]:
tickers = np.random.choice(assets_data.columns, size=50, replace=False)

In [ ]:
assets_close_price = assets_data[tickers]

In [ ]:
assets_pct_change = assets_close_price.pct_change().dropna()

In [ ]:
covariance_annualized = assets_pct_change.cov()*np.sqrt(252)
returns_annualized = assets_pct_change.mean()*252

In [ ]:
q = 0.5
b = 12
return_risk_free = 0.0375
weights_array = np.array([1/b for _ in range(len(tickers))])

In [ ]:
model = Model(name="combinatorial_portfolio_optimization")
x = np.array([model.binary_var(name=f"x({i})") for i in range(len(tickers))])
model.minimize(q*((x*weights_array).T@covariance_annualized.values@(x*weights_array))-(1-q)*(returns_annualized.values@(x*weights_array))+return_risk_free)
model.add_constraint(x.sum() == b)
print(model.prettyprint())

In [ ]:
result = model.solve()

In [ ]:
result.objective_value

In [ ]:
result_array = np.array([result.get_value(f"x({i})") for i in range(assets_close_price.columns.shape[0])])

In [ ]:
result_array

In [ ]:
quad_model = from_docplex_mp(model=model)
print(quad_model.prettyprint())

In [ ]:
qubo_converter = OptimizationProblemToQubo(penalty=1e-12)
qubo = qubo_converter.convert(quad_model)
print(qubo.prettyprint())

In [ ]:
ising, offset = qubo.to_ising()

In [ ]:
qc = DickeStateAnsatz().generate_quantum_circuit(n=len(tickers), k=b, measurement=False)
qc = pm.run(qc)

In [ ]:
x0 = 4*np.pi*np.random.random(size=qc.num_parameters)
bounds = [(0, 4*np.pi) for _ in range(x0.shape[0])]
result = scipy.optimize.minimize(fun=objective_scipy, x0=x0, method="cobyla", bounds=bounds, args=(estimator, qc, ising, offset))

In [ ]:
result

In [ ]:
ansatz = qc.copy()
params_mapper = {param: value for param, value in zip(ansatz.parameters, result.x)}
ansatz = ansatz.assign_parameters(parameters=params_mapper)
ansatz.measure_all()

In [ ]:
counts = simulator.run(circuits=ansatz, shots=4096).result().get_counts()

In [ ]:
objective_optuna = ObjectiveOptuna(qc=qc, estimator=estimator, ising=ising, offset=offset)

In [ ]:
sampler = optuna.samplers.CmaEsSampler()
study = optuna.create_study(sampler=sampler)
study.optimize(objective_optuna, n_trials=100)

In [ ]:
study.best_value

In [ ]:
ansatz = qc.copy()
params_mapper = {param: value for param, value in zip(ansatz.parameters, study.best_params.values())}
ansatz = ansatz.assign_parameters(parameters=params_mapper)
ansatz.measure_all()

In [ ]:
counts = simulator.run(circuits=ansatz, shots=4096).result().get_counts()

In [ ]:
counts